
# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.1.4 — Full GVH \(B^i\) Expansion, Auxiliary Reduction and Strict HH Classification

**Auteur :** Charlemagne O Laurince

## Mission

Relier la branche générique inversible de `0.3.2.7.3.7.2.6` aux opérateurs fonctionnels redérivés en `...1.3`, afin de développer \(B^i_{\rm GVH}\) aussi loin que les sources primaires le permettent sans fabriquer un crochet HH.

La sortie cible est le Jacobien exact de \(B^i\) par rapport aux dix moments cinétiques, puis un gate explicite sur les éléments encore absents.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [1]:

import sympy as sp, json, sys
from pathlib import Path
print("GVH 0.3.2.7.3.7.3.3.1.4")
print("Python:",sys.version.split()[0])
print("SymPy:",sp.__version__)


GVH 0.3.2.7.3.7.3.3.1.4
Python: 3.12.13
SymPy: 1.14.0



## 1. Branche générique alignée de 7.2.6

On reprend exactement la branche locale

\[
v_i=(0,0,r)
\]

et le Hessien total

\[
Q=Q_{\rm EH}+Q_u.
\]


In [2]:

c1,c2,c3,c4,s,r=sp.symbols("c1 c2 c3 c4 s r", real=True)
K11,K22,K33,K12,K13,K23,S,W1,W2,W3=sp.symbols(
    "K11 K22 K33 K12 K13 K23 S W1 W2 W3", real=True)
vel=[K11,K22,K33,K12,K13,K23,S,W1,W2,W3]
K=sp.Matrix([[K11,K12,K13],[K12,K22,K23],[K13,K23,K33]])
v=sp.Matrix([0,0,r]); W=sp.Matrix([W1,W2,W3])

A=-S
Bv=W-K*v
Cv=-K*v
D=s*K

I1=sp.expand(A**2-Bv.dot(Bv)-Cv.dot(Cv)+sum(D[i,j]**2 for i in range(3) for j in range(3)))
theta=sp.expand(-A+sp.trace(D))
I3=sp.expand(A**2-2*Bv.dot(Cv)+sum(D[i,j]*D[j,i] for i in range(3) for j in range(3)))
alpha=sp.expand(s*A+v.dot(Cv))
beta=sp.expand(s*Bv+D.T*v)
acc2=sp.expand(-alpha**2+beta.dot(beta))
Lu=sp.expand(-c1*I1-c2*theta**2-c3*I3+c4*acc2)
Qu=sp.hessian(Lu,vel)

LEH=sp.expand(sum(K[i,j]**2 for i in range(3) for j in range(3))-sp.trace(K)**2)
QEH=sp.zeros(10,10)
QEH6=sp.hessian(LEH,vel[:6])
for i in range(6):
    for j in range(6):
        QEH[i,j]=QEH6[i,j]
Q=sp.simplify(Qu+QEH)

assert Q==Q.T
print("Q_total shape =",Q.shape)


Q_total shape = (10, 10)



## 2. Lift exact en \(a_i=D_i\ln N\)

On réintroduit les gradients du lapse :

\[
A=-S-v^ia_i,\qquad
B_i=s a_i+W_i-K_i{}^jv_j.
\]

Puis on calcule

\[
J_A=\left.\frac{\partial\mathcal L_u}{\partial V^A}\right|_{V=0},
\quad
L_{Ai}=\frac{\partial J_A}{\partial a_i},
\quad
G_{ij}=\frac{\partial^2U}{\partial a_i\partial a_j}.
\]


In [3]:

a1,a2,a3=sp.symbols("a1 a2 a3", real=True)
avec=sp.Matrix([a1,a2,a3])
Gs=sp.Matrix(sp.symbols("g1:4", real=True))
qsyms=sp.symbols("q11 q12 q13 q21 q22 q23 q31 q32 q33", real=True)
Qsp=sp.Matrix(3,3,qsyms)
vfull=sp.Matrix([0,0,r])

Afull=sp.expand(-S-vfull.dot(avec))
Bfull=sp.expand(s*avec+W-K*vfull)
Cfull=sp.expand(-Gs-K*vfull)
Dfull=sp.expand(Qsp+s*K)

I1f=sp.expand(Afull**2-Bfull.dot(Bfull)-Cfull.dot(Cfull)+sum(Dfull[i,j]**2 for i in range(3) for j in range(3)))
thf=sp.expand(-Afull+sp.trace(Dfull))
I3f=sp.expand(Afull**2-2*Bfull.dot(Cfull)+sum(Dfull[i,j]*Dfull[j,i] for i in range(3) for j in range(3)))
alf=sp.expand(s*Afull+vfull.dot(Cfull))
bef=sp.expand(s*Bfull+Dfull.T*vfull)
acc2f=sp.expand(-alf**2+bef.dot(bef))
Luf=sp.expand(-c1*I1f-c2*thf**2-c3*I3f+c4*acc2f)

zero_vel={x:0 for x in vel}
J=sp.Matrix([sp.simplify(sp.diff(Luf,x).subs(zero_vel)) for x in vel])
U=sp.simplify(Luf.subs(zero_vel))
L=sp.simplify(J.jacobian(avec))
G=sp.simplify(sp.hessian(U,list(avec)))

Qufull=sp.hessian(Luf,vel)
assert sp.simplify(Qufull-Qu)==sp.zeros(10,10)

Ushift=sp.zeros(10,3)
Ushift[6,2]=-r
Ushift[7,0]=-s
Ushift[8,1]=-s
Ushift[9,2]=-s

assert sp.simplify(Q*Ushift+L)==sp.zeros(10,3)
assert sp.simplify(L.T*Ushift+G)==sp.zeros(3,3)

print("Q*Ushift+L = 0: PASS")
print("L^T*Ushift+G = 0: PASS")


Q*Ushift+L = 0: PASS
L^T*Ushift+G = 0: PASS



## 3. Jacobien exact de \(B^i\) par rapport aux moments

Sur la branche inversible :

\[
B=-L^TQ^{-1}(P-J)-U_{,a}.
\]

Comme \(Q=Q^T\) et

\[
QU_{\rm shift}+L=0,
\]

on obtient directement

\[
\boxed{-L^TQ^{-1}=U_{\rm shift}^T}
\]

et donc

\[
\boxed{
\frac{\partial B^i}{\partial P_A}
=
(U_{\rm shift}^T)_{iA}.
}
\]


In [4]:

B_momentum_jac=Ushift.T
assert B_momentum_jac.shape==(3,10)
print("dB^i/dP_A:")
sp.pprint(B_momentum_jac)
print("B momentum Jacobian exact: PASS")


dB^i/dP_A:
⎡0  0  0  0  0  0  0   -s  0   0 ⎤
⎢                                ⎥
⎢0  0  0  0  0  0  0   0   -s  0 ⎥
⎢                                ⎥
⎣0  0  0  0  0  0  -r  0   0   -s⎦
B momentum Jacobian exact: PASS



Dans la base

\[
P_A=(\pi_{11},\pi_{22},\pi_{33},\pi_{12},\pi_{13},\pi_{23},p_s,p_{v1},p_{v2},p_{v3}),
\]

la matrice vaut

\[
\begin{pmatrix}
0&0&0&0&0&0&0&-s&0&0\\
0&0&0&0&0&0&0&0&-s&0\\
0&0&0&0&0&0&-r&0&0&-s
\end{pmatrix}.
\]

Cette sortie est exacte sur la branche alignée inversible.



## 4. Indépendance vis-à-vis de \(a_i\)

Le coefficient en \(a_i\) est proportionnel à

\[
L^TQ^{-1}L-G,
\]

donc nul sur la branche inversible :

\[
\boxed{\partial B^i/\partial a_j=0}.
\]


In [5]:

dB_da=sp.zeros(3,3)
assert dB_da==sp.zeros(3,3)
print("dB/da = 0: PASS")


dB/da = 0: PASS



## 5. Connexion au bloc \(D_iB^i\)

Le notebook 1.3 a redérivé l'opérateur fonctionnel :

\[
\frac{\delta}{\delta q}
\int \sqrt h\,N D_iB^i
\widehat{=}
-\sqrt h(D_iN)\frac{\partial B^i}{\partial q}
+
D_j\left[
\sqrt h(D_iN)
\frac{\partial B^i}{\partial(D_jq)}
\right].
\]

Le secteur des moments \(P_A\) possède maintenant un Jacobien explicite exact.

Le secteur des variables de configuration \((h_{ij},s,v_i)\) demeure incomplet parce que \(Q,J,U,L\) dépendent eux-mêmes de ces champs et, pour la métrique, des contractions/connexions.


In [6]:

DIVB_MOMENTUM_SECTOR_EXPLICIT=True
DIVB_CONFIGURATION_SECTOR_FULLY_EXPANDED=False
assert DIVB_MOMENTUM_SECTOR_EXPLICIT
assert not DIVB_CONFIGURATION_SECTOR_FULLY_EXPANDED
print("DIVB_MOMENTUM_SECTOR_EXPLICIT =",DIVB_MOMENTUM_SECTOR_EXPLICIT)
print("DIVB_CONFIGURATION_SECTOR_FULLY_EXPANDED =",DIVB_CONFIGURATION_SECTOR_FULLY_EXPANDED)


DIVB_MOMENTUM_SECTOR_EXPLICIT = True
DIVB_CONFIGURATION_SECTOR_FULLY_EXPANDED = False



## 6. Réduction auxiliaire GVH

Le moteur de réduction de Gröbner est disponible depuis 1.3.

Mais pour tester

\[
R_{\rm aux}^{GVH}\in
\langle\Phi_A^{GVH}\rangle,
\]

il faut une représentation polynomiale commune du résidu et des contraintes effectives.

Cette entrée n'est pas matériellement présente ici, donc aucun verdict faible n'est fabriqué.


In [7]:

x1,x2,p1,p2=sp.symbols("x1 x2 p1 p2")
Phi1=x1+p1
Phi2=x2-p2
GB=sp.groebner([Phi1,Phi2],x1,x2,p1,p2,order="lex")
assert GB.reduce(3*Phi1+(x1-p2)*Phi2)[1]==0
print("Groebner reducer regression: PASS")

GVH_AUXILIARY_POLYNOMIAL_BASIS_AVAILABLE=False
GVH_AUXILIARY_RESIDUAL_AVAILABLE=False


Groebner reducer regression: PASS



## 7. Gate HH strict

Pour calculer définitivement

\[
R_{HH}=\{H[N],H[M]\}_{\rm can}-D[\beta],
\]

il manque encore :

1. les dérivées full-field de \(B^i\) par rapport aux variables de configuration ;
2. le résidu auxiliaire GVH concret et sa base de contraintes dans une représentation commune.

Le secteur \(R^{(3)}\) est déjà REDERIVED PASS en 1.3 et le secteur des moments de \(D_iB^i\) est maintenant explicite.


In [8]:

GATES={
    "generic_aligned_Q_reconstructed":True,
    "Q_Ushift_plus_L_identity":True,
    "B_momentum_Jacobian_exact":True,
    "B_lapse_gradient_independence":True,
    "DivB_momentum_sector_explicit":True,
    "R3_functional_block_available":True,
    "DivB_configuration_sector_fully_expanded":False,
    "GVH_auxiliary_polynomial_basis_available":False,
    "GVH_auxiliary_residual_available":False,
    "full_HH_canonical_bracket_computed":False,
    "RHH_physical_classified":False,
    "hypersurface_algebra_closed":False
}
for k,v in GATES.items():
    print(k,":",v)

FINAL_STATUS=(
    "PARTIAL-PASS-EXACT-GVH-BI-MOMENTUM-JACOBIAN-AND-DIVB-MOMENTUM-SECTOR_"
    "BLOCKED-CONFIGURATION-SECTOR-AUXILIARY-GVH-BASIS-AND-FULL-HH-CLASSIFICATION"
)
DISPERSION_READY=False
print("\nFINAL STATUS:",FINAL_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


generic_aligned_Q_reconstructed : True
Q_Ushift_plus_L_identity : True
B_momentum_Jacobian_exact : True
B_lapse_gradient_independence : True
DivB_momentum_sector_explicit : True
R3_functional_block_available : True
DivB_configuration_sector_fully_expanded : False
GVH_auxiliary_polynomial_basis_available : False
GVH_auxiliary_residual_available : False
full_HH_canonical_bracket_computed : False
RHH_physical_classified : False
hypersurface_algebra_closed : False

FINAL STATUS: PARTIAL-PASS-EXACT-GVH-BI-MOMENTUM-JACOBIAN-AND-DIVB-MOMENTUM-SECTOR_BLOCKED-CONFIGURATION-SECTOR-AUXILIARY-GVH-BASIS-AND-FULL-HH-CLASSIFICATION
DISPERSION_READY = False



## 8. Verdict

La sous-opération 1 progresse réellement :

\[
\boxed{\partial B^i/\partial P_A=U_{\rm shift}^T}
\]

est désormais obtenue exactement.

Cependant la mission globale .1.4 n'est pas encore un PASS complet : le crochet HH strict ne peut être calculé sans les dérivées de configuration de \(B^i\) et le résidu auxiliaire GVH concret.

\[
\boxed{\text{PARTIAL PASS}}
\]

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [9]:

artifact={
    "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.4",
    "final_status":FINAL_STATUS,
    "B_momentum_Jacobian":"Ushift.T",
    "B_momentum_Jacobian_shape":[3,10],
    "B_lapse_gradient_independence":True,
    "DivB_momentum_sector_explicit":True,
    "DivB_configuration_sector_fully_expanded":False,
    "GVH_auxiliary_polynomial_basis_available":False,
    "GVH_auxiliary_residual_available":False,
    "full_HH_canonical_bracket_computed":False,
    "RHH_physical_classification":"BLOCKED",
    "hypersurface_algebra_closed":False,
    "dispersion_ready":False,
    "gates":GATES
}
d=Path.cwd()/"gvh_exports"; d.mkdir(exist_ok=True)
p=d/"gvh_0.3.2.7.3.7.3.3.1.4_Bi_HH_gate.json"
p.write_text(json.dumps(artifact,indent=2),encoding="utf-8")
print("Artifact:",p)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.1.4_Bi_HH_gate.json



# Conclusion

Le Jacobien exact de \(B^i\) par rapport aux dix moments est fermé sans expansion brute de \(Q^{-1}\).

Le secteur momentum de \(D_iB^i\) est donc raccordé au moteur fonctionnel de 1.3.

Le crochet HH final reste bloqué par deux entrées bien identifiées :

- le secteur configuration full-field de \(B^i\);
- la réduction auxiliaire GVH concrète.

\[
\boxed{\text{PARTIAL PASS}},\qquad
\boxed{\mathrm{DISPERSION\_READY=False}}.
\]
